In [43]:
from helpers.persistence import save_var, load_var
from helpers.progress_bar import ProgressBar
from helpers.openml_data_v2 import get_data1, openml_cc18_list, hard_list
from helpers.openml_data import tabular_id_list

In [44]:
import numpy as np
from tqdm import tqdm
import scipy

In [45]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, label_binarize
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import pairwise_distances, pairwise_distances_chunked

In [46]:
from NPT.run import main
from NPT.npt.configs import build_parser

In [47]:
# npt uses sklearn to do CV and splitting and uses the same random state = 42 and has same test_index
# problem is that somewhere in trainer or deeper the data rows are shuffled and that is the y_preds assertion fails

In [48]:
# dataset_name = 23

# parser = build_parser()
# args = parser.parse_args([
#     '--data_set', f'custom__{dataset_name}', 
#     '--custom_data_set', f'{dataset_name}', 
#     '--exp_test_perc', '0.2',
#     '--exp_val_perc', '0.1',
#     '--exp_patience', '30',
#     '--exp_n_runs', '1',
#     '--exp_num_total_steps', '10',
#     '--exp_batch_size', '128',
#     # '--exp_disable_cuda',
#     # '--data_set_on_cuda', 'True',
#     '--exp_full_batch_gd',
# ])


# fold_preds, fold_trues = main(args)
# # dataset, _ = main(args)
# # args

In [49]:
def get_cv_results(test_preds, test_trues):
    
    scores = []
    y_trues = []
    y_preds = []
    
    
    # pbar.add_prefix('starting 10-fold cv')

    for fold_index, (y_test, y_pred) in enumerate(zip(test_preds, test_trues)):    
        
        score = f1_score(y_test, y_pred, average='weighted')
        
        # acc = ((y_test == y_pred).sum() / y_test.shape[0])
        # print('acc', acc)
        
        scores.append(score)
        y_preds.append(y_pred)
        y_trues.append(y_test)
        
    return scores, y_preds, y_trues


# scores, _, _ = get_cv_results(fold_preds, fold_trues)
# scores

In [50]:
save_path = './saved_vars/test-npt.pkl'
dataset_results = load_var(save_path) or {}

cache_path = './saved_vars/test-npt-outputs.pkl'
outputs = load_var(cache_path) or {}

# dataset_results, outputs = {}, {}

In [51]:
datasets = [ 1063,  1510,  1464,   469,   458,  1494,  1068,  1049,    23,
        1050, 40975, 40982,  1067,  1487,  1485,  4134, 40701,  1497,
        1475,  4538]

In [52]:
done=list(outputs.keys())
len(done), done

(14,
 [23,
  469,
  1049,
  1063,
  1067,
  1068,
  1464,
  1510,
  1494,
  40975,
  40982,
  40701,
  1475,
  4134])

In [53]:
np.setdiff1d(datasets, done)

array([ 458, 1050, 1485, 1487, 1497, 4538])

In [54]:
import pandas as pd

rows = []
columns = ['dataset','corruption','fold','test_score']

for k, (scores, preds, trues) in dataset_results.items():
    for i, s in enumerate(scores):
        rows.append([k, 'NPT', i, s])
    
    
    # print(k, scores, len(preds), len(trues))
    # break
    
df = pd.DataFrame(rows, columns=columns)
df.to_csv('./exports/npt.csv', index=0)

In [55]:
df.dataset.unique()

array([   23,   469,  1049,  1063,  1067,  1068,  1464,  1510,  1494,
       40975, 40982, 40701,  1475])